# STAT 764 · Meeting 10 — Threshold laboratory

**Thursday, October 8**

On Tuesday the model gave every discharged patient a probability of coming back within
thirty days, and the default rule — flag anyone at 0.5 or above — flagged almost nobody.

A classifier is two things: **a probability, and a rule for acting on it.** Tuesday built
the first. Today is the second, and the model does not change once all meeting.

This is not a technicality. A hospital does not act on a probability; it calls a patient,
sends a nurse, or enrolls someone in a program, and every one of those costs money and
helps some of the people it reaches. Where you draw the line decides who gets the call.

**The question for today.** The model gives every patient a probability. Where do you draw
the line — and what should decide it: a metric, or the decision you are making?

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Tuesday's model, rebuilt so this notebook stands on its own.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]  # death, hospice
y_all = (enc["readmitted"] == "<30").astype(int)

NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
X_all = enc[NUM + CAT].astype({c: str for c in CAT})

# Split by PATIENT, so nobody is on both sides.
train, test = next(GroupShuffleSplit(1, test_size=0.25, random_state=764)
                   .split(X_all, y_all, groups=enc["patient_nbr"]))


def readmission_model(**options):
    """Tuesday's pipeline. `options` go to LogisticRegression."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000, **options)),
    ])


model = readmission_model().fit(X_all.iloc[train], y_all.iloc[train])
p = model.predict_proba(X_all.iloc[test])[:, 1]      # one probability per discharge
y = y_all.iloc[test].to_numpy()                       # 1 = back within 30 days

patients = enc["patient_nbr"].iloc[test].nunique()
print(f"  {len(y):,} discharges of {patients:,} patients the model never saw")
print(f"  readmitted within 30 days: {y.sum():,} ({y.mean():.1%})")
print(f"  AUC {roc_auc_score(y, p):.3f} · probabilities from {p.min():.3f} to {p.max():.3f}")

## 0. Finishing Tuesday

### Precision and prevalence, in two tables

Tuesday showed that the same model, at the same threshold, has very different precision in
different groups of patients. Here is why, with nothing but counting.

Take 1,000 patients and a model with **sensitivity 80%** and **specificity 90%**.
**Prevalence decides how many patients are in each row.** Sensitivity splits the *came back*
row into flagged and not flagged; specificity splits the *did not* row. Precision then reads
**down the flagged column**.

In [ ]:
def four_boxes(prevalence, sensitivity, specificity, n=1000):
    """n patients. Prevalence fills the two rows; sensitivity and specificity split each row."""
    came_back, did_not = n * prevalence, n * (1 - prevalence)      # the rows: prevalence only
    boxes = pd.DataFrame(
        {"flagged":     [sensitivity * came_back, (1 - specificity) * did_not],
         "not flagged": [(1 - sensitivity) * came_back, specificity * did_not]},
        index=["came back", "did not come back"])
    boxes["row total"] = boxes.sum(axis=1)
    boxes.loc["column total"] = boxes.sum()
    return boxes


# The same model -- the same two rates -- in two populations. Only the rows change.
for prevalence in (0.05, 0.30):
    boxes = four_boxes(prevalence, sensitivity=0.80, specificity=0.90)
    caught, flagged = boxes.loc["came back", "flagged"], boxes.loc["column total", "flagged"]
    print(f"prevalence {prevalence:.0%}: 1,000 patients, sensitivity 80%, specificity 90%")
    print(boxes.round().astype(int).to_string())
    print(f"  precision = {caught:.0f} / {flagged:.0f} = {caught / flagged:.0%}\n")

**The model is the same in both tables — 80% of the top row flagged, 10% of the bottom row
flagged. Only the row totals changed, and precision went from 30% to 77%.** The flagged
column mixes the two rows: 40 caught beside 95 false alarms when 5% come back, because 10%
of a 950-patient row is still 95 people; 240 caught beside 70 false alarms when 30% do. A
rare outcome means a huge bottom row, and even a small false-alarm rate floods the column.

**In counts — the table as a formula.** Precision reads down the flagged column:

$$\begin{aligned}
\text{precision} &= \frac{\text{flagged and came back}}{\text{everyone flagged}} \\[10pt]
  &= \frac{\text{sensitivity} \times \text{came back}}{\text{sensitivity} \times \text{came back} \;+\; (1-\text{specificity}) \times \text{did not}} \\[10pt]
  &= \frac{0.80 \times 50}{0.80 \times 50 \;+\; 0.10 \times 950} \;=\; \frac{40}{40 + 95} \;=\; 0.30
\end{aligned}$$

*came back* and *did not* are the row totals — the only place the prevalence enters.

**In probabilities — divide every count by the 1,000 patients.** The row totals become π and
1 − π, and the formula becomes **Bayes' rule** in its usual form. Write $D$ for *came back* and
$+$ for *flagged*:

$$\begin{aligned}
P(D \mid +) &= \frac{P(+ \mid D)\,P(D)}{P(+)} \\[10pt]
            &= \frac{P(+ \mid D)\,P(D)}{P(+ \mid D)\,P(D) \;+\; P(+ \mid \text{not } D)\,P(\text{not } D)}
\end{aligned}$$

The second line expands $P(+)$ by the law of total probability: everyone flagged either came
back or did not. Every piece is something you already have a name for — checked here against the 5% table:

| conditional probability | its name | in the 5% table |
|---|---|---|
| $P(D \mid +)$ | **precision** | 40 / 135 = 0.30 |
| $P(+ \mid D)$ | **sensitivity** | 40 / 50 = 0.80 |
| $P(+ \mid \text{not } D)$ | **1 − specificity**, the false positive rate | 95 / 950 = 0.10 |
| $P(D)$ | **prevalence**, π | 50 / 1,000 = 0.05 |
| $P(+)$ | the share flagged | 135 / 1,000 = 0.135 |

Put the names in:

$$\begin{aligned}
\text{precision} &= \frac{\text{sensitivity}\cdot \pi}{\text{share flagged}} \\[10pt]
                 &= \frac{\text{sensitivity}\cdot \pi}{\text{sensitivity}\cdot \pi \;+\; (1-\text{specificity})\,(1-\pi)}
\end{aligned}$$

The top is the flagged-and-came-back box; the bottom is the whole flagged column. Hold
sensitivity and specificity fixed, and π is the only thing left that can move precision.
Here it is for Tuesday's model:

In [ ]:
# Tuesday's rule -- threshold 0.2 -- on the same patients the model never saw.
flag = p >= 0.2
sens = flag[y == 1].mean()                   # P(flagged | came back)
spec = (~flag)[y == 0].mean()                # P(not flagged | did not come back)
print(f"  sensitivity {sens:.3f} · specificity {spec:.3f}\n")


def precision_at(prevalence):
    """Bayes' rule: precision from sensitivity, specificity and the base rate -- nothing else."""
    return sens * prevalence / (sens * prevalence + (1 - spec) * (1 - prevalence))


# Check the formula against these patients, where every box is counted.
print(f"  measured precision on these patients (prevalence {y.mean():.1%})   {y[flag].mean():.3f}")
print(f"  Bayes' rule at the same prevalence                         {precision_at(y.mean()):.3f}\n")

# Tuesday's two groups: patients who appear once in the data, and five or more times.
times = enc.groupby("patient_nbr")["patient_nbr"].transform("size").clip(upper=5)
once, often = y_all[times == 1].mean(), y_all[times == 5].mean()

# Now hold sensitivity and specificity fixed and change only the population.
for prevalence in (0.01, once, y.mean(), often, 0.50):
    print(f"  prevalence {prevalence:6.1%}   ->   precision {precision_at(prevalence):6.1%}")

**Same model, same threshold — precision from about 3% to about 73%,** and only the
population changed. On these patients Bayes' rule gives back the measured precision exactly;
it is arithmetic, not an approximation. The 4.3% and 28.3% rows are Tuesday's patients seen
once and patients seen five or more times. Tuesday's breast-ultrasound table — precision 2%,
22% and 58% in screening, diagnostic and referral populations — was this same computation.

⚠ **The *if*.** All of this assumes sensitivity and specificity stay the same in every
population. Tuesday's studio question 2 tested exactly that, on patients with and without
earlier hospital stays.

### Why AUC is the chance that a pair is ranked right

*In class this is a paper worksheet, laptops closed: draw the curve, shade it, count pairs.
The cell below is the check — run it afterwards.*

Tuesday's model had AUC 0.655, which means: pick one patient who came back and one who did
not, and the model gives the one who came back the higher probability 65.5% of the time.
Why should the *area under a curve* be that probability? Ten made-up patients, sorted from
highest probability to lowest, show it.

Walk down the list, lowering the threshold past one patient at a time. A patient who came
back moves the curve one step **up** (one more readmission caught); a patient who did not
moves it one step **right** (one more false alarm). That path is the **ROC curve**.

In [ ]:
# Ten made-up patients, sorted from highest probability to lowest.
toy = pd.DataFrame({"probability": [0.92, 0.81, 0.74, 0.66, 0.55, 0.43, 0.35, 0.28, 0.17, 0.08],
                    "came back":   [1,    1,    0,    1,    0,    0,    1,    0,    0,    0]})
came, did_not = toy["came back"].sum(), (1 - toy["came back"]).sum()          # 4 and 6

# Lower the threshold past one patient at a time: came back -> one step UP (1/4 of the way),
# did not -> one step RIGHT (1/6 of the way). The path is the ROC curve.
up = np.r_[0, toy["came back"].cumsum() / came]
right = np.r_[0, (1 - toy["came back"]).cumsum() / did_not]

# The area, one strip per patient who did NOT come back: the strip above that patient's step
# is as tall as the share of readmitted patients ranked ABOVE them.
above = toy["came back"].cumsum()[toy["came back"] == 0]
for prob, k in zip(toy.loc[toy["came back"] == 0, "probability"], above):
    print(f"  did not come back, probability {prob:.2f}:   {k} of {came} readmitted ranked above")
print(f"\n  area = average strip height = {above.mean() / came:.3f}"
      f"   =   {above.sum()} of {came * did_not} pairs in the right order")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(right, up, marker="o", label=f"ten patients, AUC {above.mean() / came:.2f}")
ax.fill_between(right, up, alpha=0.15)
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="a coin, AUC 0.5")
ax.set_xlabel("false positive rate (one step right = one false alarm)")
ax.set_ylabel("sensitivity (one step up = one readmission caught)")
ax.legend(loc="lower right")
plt.show()

**Read the area one strip at a time.** Each step to the right is one patient who did not
come back, and the strip above it is as tall as the share of readmitted patients ranked
**above** that patient: 2 of 4, then 3, 3, 4, 4 and 4. So the area is the average of those
shares — which is the share of all (came back, did not) pairs the model puts in the right
order: **20 of 24, 0.83**. A coin gets half the pairs right and runs along the diagonal; a
perfect ranking gets every pair right and fills the square.

Tuesday's 0.655 is the same count over 2,739 readmitted patients and 21,967 who were not.

From here on, **`p` and `y` do not change.** Every number below is a different rule
applied to the same probabilities.

## 1. Ask a metric for a threshold

Any metric you know can be turned into a threshold: try every cutoff, keep the one the
metric likes best. Here are four rules you could defend in a meeting, each asked the same
question about the same model.

In [ ]:
cutoffs = np.round(np.arange(0.005, 0.95, 0.001), 3)


def best_cutoff(metric):
    """The cutoff this metric scores highest, over every cutoff on the grid."""
    scores = [metric(y, p >= c) for c in cutoffs]
    return cutoffs[int(np.argmax(scores))]


rules = {"the default, 0.5": 0.5,
         "best accuracy": best_cutoff(accuracy_score),
         "best F1": best_cutoff(f1_score),
         "best balanced accuracy": best_cutoff(balanced_accuracy_score)}

rows = []
for rule, c in rules.items():
    flag = p >= c
    rows.append({"rule": rule, "threshold": c, "patients flagged": int(flag.sum()),
                 "share flagged": f"{flag.mean():.1%}",
                 "readmissions caught": int((flag & (y == 1)).sum())})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n  accuracy of the best-accuracy cutoff {accuracy_score(y, p >= rules['best accuracy']):.4f}"
      f" -- flagging nobody at all scores {1 - y.mean():.4f}")

Four reasonable rules, and they flag anywhere from a couple of dozen patients to
thousands. **None of them is wrong about its own metric. None of them knows what a flag
costs, or what it buys.**

Accuracy's answer deserves a second look: its best cutoff flags almost nobody, and it
beats flagging *nobody at all* by a hair. With eleven percent of patients coming back,
the most accurate thing to do is very nearly nothing.

## 2. What a flag is worth

Flagging a patient triggers something — say a nurse's home visit. Whether a visit is worth
making depends on **four numbers**:

| symbol | what it is | today's example |
|---|---|---|
| **p** | the model's probability that *this* patient comes back within 30 days | e.g. 0.30 |
| **c** | what one visit costs | \$150 |
| **e** | how effective a visit is — the share of would-be readmissions it prevents | 5%, 1 in 20 |
| **R** | what one readmission costs | \$17,700 |

\$17,700 is the average adult 30-day readmission in 2020 (AHRQ HCUP Statistical Brief #307); the
\$150 and the 5% are illustrative.

**Step 1 — a patient who will certainly come back** (p = 1). Visit 20 such patients and the visits
prevent 1 readmission (5% of 20), saving \$17,700. Per visit, that is **e × R = 0.05 × \$17,700 =
\$885** — the most one visit can ever save.

**Step 2 — a patient who *might* come back** (probability p). Only a share p of patients like
this one would come back at all, so only that share of the visits has anything to prevent. The
expected saving per visit is **p × e × R**. "Expected" means the average over many such patients:
visit 1,000 patients at p = 0.30, and 300 of them would come back; the visits prevent 5% of those
— 15 readmissions — saving 15 × \$17,700 = \$265,500, which is **\$265.50 per visit**
(= 0.30 × 0.05 × \$17,700).

**Step 3 — compare with what the visit costs.** Visit when the expected saving beats the cost:

$$p \cdot e \cdot R \;>\; c$$

At p = 0.30: \$265.50 against \$150 — **visit**, and gain \$115.50 on average. At p = 0.10: \$88.50
against \$150 — **don't**; it would lose \$61.50.

**Step 4 — the break-even probability.** Set the two sides equal and solve for p:

$$p^* \;=\; \frac{c}{e \cdot R} \;=\; \frac{150}{0.05 \times 17{,}700} \;=\; \frac{150}{885} \;=\; 0.169$$

**Flag every patient whose probability is above 0.169.** The threshold is a ratio of prices —
nothing in it comes from the model; the model only supplies each patient's p.

In [ ]:
READMISSION = 17_700            # dollars — AHRQ HCUP Statistical Brief #307 (2024), 2020 data


def net_per_1000(threshold, cost, prevents, probs=p):
    """Dollars saved minus dollars spent, per 1,000 discharges, flagging probs >= threshold."""
    flag = probs >= threshold
    saved = (flag & (y == 1)).sum() * prevents * READMISSION
    spent = flag.sum() * cost
    return (saved - spent) / len(y) * 1000


cost, prevents = 150, 0.05
p_star = cost / (prevents * READMISSION)
print(f"  break-even threshold p* = {cost} / ({prevents} x {READMISSION:,}) = {p_star:.3f}\n")

candidates = {"break-even p*": p_star, "the default, 0.5": 0.5,
              "best F1": rules["best F1"],
              "best balanced accuracy": rules["best balanced accuracy"],
              "visit everyone": 0.0, "visit no one": 1.01}
for rule, c in candidates.items():
    print(f"  {rule:<24} flags {np.mean(p >= c):6.1%}   net ${net_per_1000(c, cost, prevents):>9,.0f}"
          " per 1,000 discharges")

grid = np.linspace(0.0, 0.6, 241)
plt.figure(figsize=(7, 3.6))
plt.plot(grid, [net_per_1000(c, cost, prevents) for c in grid])
plt.axhline(0, color="gray", lw=0.8)
plt.axvline(p_star, ls="--", color="black", lw=1)
plt.xlabel("threshold")
plt.ylabel("net $ per 1,000 discharges")
plt.title("$150 visit, prevents 5%: the same model at every threshold")
plt.tight_layout()
plt.show()

The break-even rule earns the most of the six, and it needed no search: two prices and a
division. F1's threshold earns about half as much; balanced accuracy's loses money;
visiting everyone loses a great deal.

⚠ This is the same model in every row. **The threshold is not a property of the model —
it is a property of the decision.**

## 3. The rule assumes the probabilities are true

`p > p*` compares a probability with a price ratio. That only works if a patient the
model calls 0.2 comes back about 20% of the time. So check: sort the discharges into ten
equal groups by predicted probability, and compare what the model said with what
happened.

This section has two parts: first a check that our model's probabilities are honest, then
what the most common fix for imbalanced data does to them.

In [ ]:
def by_decile(probs):
    group = pd.qcut(probs, 10, labels=False)
    return pd.DataFrame({"model said": [probs[group == k].mean() for k in range(10)],
                         "actually came back": [y[group == k].mean() for k in range(10)]},
                        index=pd.RangeIndex(1, 11, name="decile")).round(3)


print(by_decile(p).to_string())

Close in every decile — the top one overstates a little, 0.273 against 0.248 — so the
model's probabilities mean roughly what they say. That is why dividing two prices gave a
good threshold.

**Why this is here.** p* only works if the probabilities are honest, and the decile check
just said ours are. Now take the most common advice for imbalanced data and ask one question of
it: **does it keep the probabilities honest?** *Reweighting the classes* makes each readmission
count about eight times as much as a non-readmission while the model is fitted — only 1 in 9 come
back — so the model pays more attention to the rare outcome. It is one argument,
`class_weight="balanced"`, and it is usually recommended because it raises recall at the default
0.5. Predict before running: does AUC change? What happens to the probabilities — and to the
\$150 decision at p* = 0.169?

In [ ]:
reweighted = readmission_model(class_weight="balanced").fit(X_all.iloc[train],
                                                            y_all.iloc[train])
p_rw = reweighted.predict_proba(X_all.iloc[test])[:, 1]

print(f"  AUC                     {roc_auc_score(y, p):.3f} before, {roc_auc_score(y, p_rw):.3f} reweighted")
print(f"  rank agreement          {pd.Series(p).corr(pd.Series(p_rw), method='spearman'):.3f}")
print(f"  average probability     {p.mean():.3f} before, {p_rw.mean():.3f} reweighted"
      f"   (truth {y.mean():.3f})")
print(f"\n  at the same break-even p* = {p_star:.3f}:")
print(f"    Tuesday's model   flags {np.mean(p >= p_star):6.1%}   net"
      f" ${net_per_1000(p_star, cost, prevents):>9,.0f} per 1,000")
print(f"    reweighted model  flags {np.mean(p_rw >= p_star):6.1%}   net"
      f" ${net_per_1000(p_star, cost, prevents, probs=p_rw):>9,.0f} per 1,000\n")
print(by_decile(p_rw).to_string())

**The same ranking and very different probabilities.** Reweighting left the order of the
patients almost untouched — AUC and rank agreement barely move — and pushed every
probability up, so the break-even rule now flags nearly everyone.

The advice fixes imbalance *for a metric that ignores cost*, and breaks the one number a
cost-based decision needs. Whether a model's probabilities can be taken at face value has
a name — **calibration** — and it is Tuesday.

## 4. When the constraint is a count

Some decisions have no price per flag. They have **a fixed number of slots**: a team that
can review so many cases a week, and no more. Then the threshold is simply the score of
the last patient who fits, and the model's only job is to put the right patients at the
top — a question about ranking, which is exactly what AUC measures.

Studio question 2 is that decision.

## Studio

Three decisions, one model, until the exit ticket. For each: **write down your threshold
guess before you compute anything** — as a comment in the cell — and keep it when the number
disagrees. **Questions 1 and 2 are the ones to finish; question 3 if you have time, or at home.**
Fast or slow is fine: the notebook is due 24 hours after class. Done early? Lab 5 Parts 2 and 3
use exactly today's ideas.

1. **Screening — a pharmacist's phone call.** \$25 per call; it prevents 5% of the
   readmissions it reaches. What threshold, and what share of patients get a call? How
   much better is using the model than simply calling everyone? Then raise the price of a
   call — \$50, \$100, \$200 — and find where the model starts to earn its keep.
2. **A review queue.** A transitional-care nurse can review **5% of discharges**, no
   more. Where does that put the threshold? How many of the readmissions does the queue
   reach, and how does its precision compare with reviewing patients at random? What
   would doubling the nurse's capacity buy?
3. **An offer that costs real money.** A \$1,200 home-health program prevents 25% of the
   readmissions it reaches. Find the break-even threshold, the share flagged, and the net
   per 1,000 discharges. Then price **F1's threshold** and **the default 0.5** under the
   same program.

The \$25, \$1,200, 5% and 25% are illustrative; \$17,700 is not.

In [ ]:
# YOUR CODE HERE
# my guess at the threshold, before computing:

In [ ]:
# YOUR CODE HERE
# my guess at the threshold, before computing:

In [ ]:
# YOUR CODE HERE
# my guess at the threshold, before computing:

## Compare — at the start of Tuesday's class

We compare answers at the start of Tuesday's class, built from the notebooks you upload. So
leave these in your notebook, in words or as printed numbers:

1. Your three threshold guesses, written **before** you computed, and the three thresholds
   you found — lowest to highest. What moved them, and what did not?
2. Question 1: did the model matter at \$25 a call? At what price did it start to?
3. Question 3: what would F1's threshold have cost the program — and who would have noticed?

## Exit ticket

> Pick one of today's three decisions. Which number in it did you not know — and whom
> would you have to ask for it?

**Answer in the cell below — two or three sentences — then save this notebook and upload
it to Canvas under "Studio notebook — Day 10". Your two studio points need this
answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **Decision curve analysis** (Vickers & Elkin, 2006, *Medical Decision Making* 26:565–574). The
plot in section 2 with the prices factored out: net benefit across every threshold, drawn
against "treat everyone" and "treat no one" — built to judge clinical prediction models
by their consequences rather than their accuracy.

📗 **Uplift modeling.** A program should go to the patients it *changes*, not simply the
ones most likely to come back — a high-risk patient the visit cannot help is money spent
for nothing. Today's rule assumed the same 5% or 25% for everyone; uplift models estimate
it person by person.

🚫 **Reweighting or oversampling "to fix imbalance" before you choose a threshold.**
Section 3 measured what it does to the probabilities. Meeting 4, the leakage laboratory,
showed what oversampling before cross-validation does to the estimate: copies of a row
land on both sides of the split, and the score is inflated.